# Creates two tables

### tabel1: OBL_TRANSACTIONS_ACTOR 

Tabel, kus on kõik obl verb, root, kääne, isik (''/alati/mitte kunagi), koht (''/alati).


### tabel2: OBL_TRANSACTIONS_ACTOR_COUNTS 

Tabel, kus on iga root sõna jaoks üldine esinemiste kord, kui palju kordi on root sõna märkega alati isik (elus), kui palju kordi on root sõna märkega mitte kunagi isik (mitte elus).


In [2]:
import sqlite3
import pandas as pd
from tqdm import tqdm
import os
from common_sql import update_table, create_count_table, create_left_join_table
import sys
sys.path.append("..")
from common_display import display_db_table 

## Configuration

In [23]:
DB_DIR = "../example_data"

TRANSACTION_DB = f"{DB_DIR}/transactions.db"
ENRICHED_TRANSACTIONS_DB = f"{DB_DIR}/enriched_transactions.db"
VERB_PATTERN_DB = f"{DB_DIR}/verb_patterns.db"
PATTERN_MATCHES_DB = f"{DB_DIR}/pattern_matches.db"

TRANSACTION_HEAD = "transaction_head"
ENRICHED_TRANSACTIONS = "transaction_row"
MATCHED_PHRASES = "matched_phrases"

# vahetabel, ainult obl transaktsioonid
OBL_TRANSACTIONS = "trans_verbobl"

#  vahetabel, ainult obl transaktsioonid kus isik=alati
OBL_TRANSACTIONS_ACTOR_ALWAYS = "trans_verbobl_actor_always"

# vahetabel, ainult obl transaktsioonid kus isik=mitte kunagi
OBL_TRANSACTIONS_ACTOR_NEVER = "trans_verbobl_actor_never"

#  vajadusel lisada: vahetabel, ainult obl transaktsioonid kus koht=alati
OBL_TRANSACTIONS_LOC_ALWAYS = "trans_verbobl_loc_always"

# uus tabel, obl transactions koos isikumääruse infoga
OBL_TRANSACTIONS_ACTOR = "trans_actor"

# uus tabel, OBL_TRANSACTIONS_ACTOR põhjal root count, elus count, koht count
OBL_TRANSACTIONS_ACTOR_COUNTS = "trans_actor_eluskoht_count"

# temporary help tables for join and counts
j1,j2, c1, c2, c3, c4, c5, c6 = "join1", "join2" , "count1", "count2", "count3", "count4", "count5", "count6"

## Connect to db

In [4]:
con = sqlite3.connect(PATTERN_MATCHES_DB)
cur = con.cursor()
cur.execute(f'ATTACH DATABASE "{TRANSACTION_DB}" AS trans')
cur.execute(f'ATTACH DATABASE "{ENRICHED_TRANSACTIONS_DB}" AS entrans')
cur.execute(f'ATTACH DATABASE "{VERB_PATTERN_DB}" AS pat')

## Workflow

### Tabel 1

### I Kõik obl transaktsioonid

In [5]:
%%time

cur.execute("""DROP TABLE IF EXISTS {tbl}""".format(tbl=OBL_TRANSACTIONS))

cur.execute("""
Create table {new_table} as
SELECT distinct
    tr.head_id as head_id,
    tbl1.verb as verb_word,
    tbl1.verb_compound as verb_compound,
    tr.lemma as root_word,
    tr.feats as tr_feats,
    tr.deprel as word_deprel,
    tr.koht as koht,
    tr.elus as elus,
    '' as actor

FROM trans.{trans_head} as tbl1
join entrans.{trans} as tr
on tbl1.id = tr.head_id
where tr.deprel = 'obl'
""".format(new_table=OBL_TRANSACTIONS, trans_head=TRANSACTION_HEAD, trans=ENRICHED_TRANSACTIONS))


CPU times: user 5.21 ms, sys: 2.1 ms, total: 7.31 ms
Wall time: 13.6 ms


In [6]:
display_db_table(con, OBL_TRANSACTIONS, 10, 'head')

,head_id,verb_word,verb_compound,root_word,tr_feats,word_deprel,koht,elus,actor
0,2,toimuma,,lõpp,"com,in,sg",obl,,,
1,2,toimuma,,1.,"<?>,ord,roman",obl,,,
2,3,saama,pihta,keel,"all,com,pl",obl,,,
3,6,kulmineeruma,,purukspeksmine,"com,kom,sg",obl,,,
4,10,tulema,,sina,"ad,sg",obl,,YES,
5,11,viilima,,tund,"com,el,pl",obl,,,
6,11,viilima,,juht,"ad,com,sg",obl,,YES,
7,16,tulema,,mis,"gen,pl",obl,,,
8,23,alustama,,muusika,"com,kom,sg",obl,,,
9,25,muutuma,,mis,"el,sg",obl,,,


### II Kõik isik=alati verbidega seotud sõnad

In [7]:
cur.execute("""DROP TABLE IF EXISTS {tbl}""".format(tbl=OBL_TRANSACTIONS_ACTOR_ALWAYS))

cur.execute("""
Create table {new_table} as
SELECT distinct
    head_id,
    verb_word,
    verb_compound,
    root_word,
    phrase_case,
    deprel,
    koht,
    elus,
    'alati' as actor
FROM {tbl}
where deprel = 'obl'
and semantic_role = 'isik_alati'
""".format(new_table=OBL_TRANSACTIONS_ACTOR_ALWAYS, tbl=MATCHED_PHRASES))


In [8]:
display_db_table(con, OBL_TRANSACTIONS_ACTOR_ALWAYS, 10, 'head')

,head_id,verb_word,verb_compound,root_word,phrase_case,deprel,koht,elus,actor
0,179,nõudma,,mina,abl,obl,,YES,alati
1,179,nõudma,tagasi,mina,abl,obl,,YES,alati
2,179,nõudma,välja,mina,abl,obl,,YES,alati
3,179,nõudma,sisse,mina,abl,obl,,YES,alati
4,179,nõudma,kokku,mina,abl,obl,,YES,alati
5,179,nõudma,juurde,mina,abl,obl,,YES,alati
6,179,nõudma,läbi,mina,abl,obl,,YES,alati
7,10,tulema,,sina,ad,obl,,YES,alati
8,86,tulema,,tema,ad,obl,,,alati
9,481,tulema,,linn,ad,obl,YES,,alati


### II Kõik isik=mitte kunagi verbidega seotud sõnad

In [9]:
cur.execute("""DROP TABLE IF EXISTS {tbl}""".format(tbl=OBL_TRANSACTIONS_ACTOR_NEVER))

cur.execute("""
Create table {new_table} as
SELECT distinct
    head_id,
    verb_word,
    verb_compound,
    root_word,
    phrase_case,
    deprel,
    koht,
    elus,
    'mitte kunagi' as actor
FROM {tbl} 
where deprel = 'obl'
and semantic_role = 'isik_mitte kunagi'
""".format(new_table=OBL_TRANSACTIONS_ACTOR_NEVER, tbl=MATCHED_PHRASES))


In [10]:
display_db_table(con, OBL_TRANSACTIONS_ACTOR_NEVER, 10, 'head')

,head_id,verb_word,verb_compound,root_word,phrase_case,deprel,koht,elus,actor
0,656,kaotama,,vana,abl,obl,,,mitte kunagi
1,428,viskama,mööda,rõdu,abl,obl,YES,,mitte kunagi
2,428,viskama,alla,rõdu,abl,obl,YES,,mitte kunagi
3,428,viskama,,rõdu,abl,obl,YES,,mitte kunagi
4,428,viskama,välja,rõdu,abl,obl,YES,,mitte kunagi
5,428,viskama,peale,rõdu,abl,obl,YES,,mitte kunagi
6,356,jooksma,,tuba,adit,obl,YES,,mitte kunagi
7,435,ajama,,selg,adit,obl,,YES,mitte kunagi
8,572,ostma,,kodu,adit,obl,YES,,mitte kunagi
9,840,minema,tagasi,kodu,adit,obl,YES,,mitte kunagi


### III Kõik koht=alati verbidega seotud sõnad

In [11]:
cur.execute("""DROP TABLE IF EXISTS {tbl}""".format(tbl=OBL_TRANSACTIONS_LOC_ALWAYS))

cur.execute("""
Create table {new_table} as
SELECT distinct
    head_id,
    verb_word,
    verb_compound,
    root_word,
    phrase_case,
    deprel,
    koht,
    elus,
    'alati' as loc
FROM {tbl}
where deprel = 'obl'
and semantic_role = 'koht_alati'
""".format(new_table=OBL_TRANSACTIONS_LOC_ALWAYS, tbl=MATCHED_PHRASES))

In [12]:
display_db_table(con, OBL_TRANSACTIONS_LOC_ALWAYS, 10, 'head')

,head_id,verb_word,verb_compound,root_word,phrase_case,deprel,koht,elus,loc
0,428,viskama,mööda,rõdu,abl,obl,YES,,alati
1,428,viskama,alla,rõdu,abl,obl,YES,,alati
2,428,viskama,,rõdu,abl,obl,YES,,alati
3,428,viskama,välja,rõdu,abl,obl,YES,,alati
4,435,ajama,,selg,adit,obl,,YES,alati
5,840,minema,tagasi,kodu,adit,obl,YES,,alati
6,53,tulema,üle,toim,adit,obl,,,alati
7,497,tulema,üle,kontserdimaja,adit,obl,,,alati
8,731,tulema,üle,kodu,adit,obl,YES,,alati
9,875,tulema,üle,kodu,adit,obl,YES,,alati


### IV Kõik obl transactionid, millel on juures info kas elus/koht ja semantic role kui vastav verb on annoteeritud

In [20]:
create_left_join_table(con, source_tbl1=OBL_TRANSACTIONS, source_tbl2=OBL_TRANSACTIONS_ACTOR_ALWAYS,result_table=j1,
                selected_columns=["tbl1.*", "tbl2.phrase_case", "tbl2.actor as actor1"],
                condition="tbl1.verb_word = tbl2.verb_word and tbl1.verb_compound = tbl2.verb_compound and tbl1.root_word = tbl2.root_word and INSTR(',' || tbl1.tr_feats || ',', ',' || tbl2.phrase_case || ',') > 0")

create_left_join_table(con, source_tbl1=j1, source_tbl2=OBL_TRANSACTIONS_ACTOR_NEVER,result_table=j2,
                selected_columns=["tbl1.*","tbl2.phrase_case as phrase_case2", "tbl2.actor as actor2"],
                condition="tbl1.verb_word = tbl2.verb_word and tbl1.verb_compound = tbl2.verb_compound and tbl1.root_word = tbl2.root_word and INSTR(',' || tbl1.tr_feats || ',', ',' || tbl2.phrase_case || ',') > 0")

create_left_join_table(con, source_tbl1=j2, source_tbl2=OBL_TRANSACTIONS_LOC_ALWAYS,result_table=OBL_TRANSACTIONS_ACTOR,
                selected_columns=["tbl1.*","tbl2.phrase_case as phrase_case3", "tbl2.loc"],
                condition="tbl1.verb_word = tbl2.verb_word and tbl1.verb_compound = tbl2.verb_compound and tbl1.root_word = tbl2.root_word and INSTR(',' || tbl1.tr_feats || ',', ',' || tbl2.phrase_case || ',') > 0")


update_table(con, OBL_TRANSACTIONS_ACTOR, "actor", "'alati'", "actor1 = 'alati'")
update_table(con, OBL_TRANSACTIONS_ACTOR, "actor", "'mitte kunagi'", "actor2 = 'mitte kunagi'")
update_table(con, OBL_TRANSACTIONS_ACTOR, "phrase_case", "phrase_case2", "phrase_case2 is not null")
update_table(con, OBL_TRANSACTIONS_ACTOR, "phrase_case", "phrase_case3", "phrase_case3 is not null and phrase_case is null")

cur.execute("""ALTER TABLE {tbl} DROP COLUMN actor1;""".format(tbl=OBL_TRANSACTIONS_ACTOR))
cur.execute("""ALTER TABLE {tbl} DROP COLUMN actor2;""".format(tbl=OBL_TRANSACTIONS_ACTOR))
cur.execute("""ALTER TABLE {tbl} DROP COLUMN phrase_case2;""".format(tbl=OBL_TRANSACTIONS_ACTOR))
cur.execute("""ALTER TABLE {tbl} DROP COLUMN phrase_case3;""".format(tbl=OBL_TRANSACTIONS_ACTOR))
cur.execute("""ALTER TABLE {tbl} DROP COLUMN tr_feats;""".format(tbl=OBL_TRANSACTIONS_ACTOR))
con.commit()

In [21]:
display_db_table(con, OBL_TRANSACTIONS_ACTOR, 20, 'head')

,head_id,verb_word,verb_compound,root_word,word_deprel,koht,elus,actor,phrase_case,loc
0,2,toimuma,,lõpp,obl,,,mitte kunagi,in,alati
1,2,toimuma,,1.,obl,,,,None,None
2,3,saama,pihta,keel,obl,,,,None,None
3,6,kulmineeruma,,purukspeksmine,obl,,,,None,None
4,10,tulema,,sina,obl,,YES,alati,ad,None
5,11,viilima,,tund,obl,,,,None,None
6,11,viilima,,juht,obl,,YES,,None,None
7,16,tulema,,mis,obl,,,,None,None
8,23,alustama,,muusika,obl,,,,None,None
9,25,muutuma,,mis,obl,,,,None,None


## Tabel 2

In [24]:
# base table with root counts
cur.execute("""drop table if exists {tbl}""".format(tbl=c1))

cur.execute("""
create table {new_table} as
SELECT root_word, count(root_word) as root_cnt
from {isik}
group by root_word
""".format(new_table=c1, isik=OBL_TRANSACTIONS_ACTOR))

# table counts elus 
create_count_table(con, OBL_TRANSACTIONS_ACTOR, c2,
                  ["root_word"], "root_word", "elus_cnt", "actor = 'alati'", ["root_word"])

# tbl count mitte elus
create_count_table(con, OBL_TRANSACTIONS_ACTOR, c3,
                  ["root_word"], "root_word", "mitte_elus_cnt", "actor = 'mitte kunagi'", ["root_word"])

# tbl count koht
create_count_table(con, OBL_TRANSACTIONS_ACTOR, c5,
                  ["root_word"], "root_word", "koht_cnt", "loc = 'alati'", ["root_word"])


In [28]:
# join everything into 1 table

create_left_join_table(con, source_tbl1=c1, source_tbl2=c2, 
        result_table=c4,
        selected_columns=["tbl1.root_word", "tbl1.root_cnt", "elus_cnt"],
        condition="tbl1.root_word=tbl2.root_word ")

create_left_join_table(con, source_tbl1=c4, source_tbl2=c3, 
        result_table=c6,
        selected_columns=["tbl1.root_word", "tbl1.root_cnt", "tbl1.elus_cnt", "mitte_elus_cnt"],
        condition="tbl1.root_word=tbl2.root_word ")

create_left_join_table(con, source_tbl1=c6, source_tbl2=c5, 
        result_table=OBL_TRANSACTIONS_ACTOR_COUNTS,
        selected_columns=["tbl1.*", "koht_cnt"],
        condition="tbl1.root_word=tbl2.root_word ")

In [29]:
# update table null -> 0
update_table(con, OBL_TRANSACTIONS_ACTOR_COUNTS, "elus_cnt", 0, "elus_cnt is null")
update_table(con, OBL_TRANSACTIONS_ACTOR_COUNTS, "mitte_elus_cnt", 0, "mitte_elus_cnt is null")
update_table(con, OBL_TRANSACTIONS_ACTOR_COUNTS, "koht_cnt", 0, "koht_cnt is null")

In [30]:
display_db_table(con, OBL_TRANSACTIONS_ACTOR_COUNTS, 8, 'head')

,root_word,root_cnt,elus_cnt,mitte_elus_cnt,koht_cnt
0,1.,1,0,0,0
1,Beatrice,2,0,0,0
2,Eesti,1,0,1,0
3,Gunnar,2,0,0,0
4,Harvard,1,0,0,0
5,Juhan,1,0,0,0
6,Lisal,1,0,0,0
7,Los,1,0,0,0


In [74]:
# example result table with more data
#query = """SELECT * from  {tbl} where elus_cnt!=mitte_elus_cnt limit 20 """.format(tbl=OBL_TRANSACTIONS_ACTOR_COUNTS)
#source2 = pd.read_sql_query(query, con)
#source2

,root_word,root_cnt,elus_cnt,mitte_elus_cnt
0,$,321,0,1
1,$1,2,0,1
2,%,32,0,4
3,%-ilis,1,0,1
4,%-põhimõte,1,0,1
5,%-see,3,0,2
6,%line,54,8,34
7,-30s,1,0,1
8,-4%,17,0,2
9,-4.,3,0,1


Delete temporary tables

In [31]:
for tbl in [j1,j2, c1, c2, c3, c4, c5, c6]:
    cur.execute("""DROP TABLE IF EXISTS {table}""".format(table = tbl))

Save table to csv if necessary

In [82]:
#query = """SELECT * from {tbl}""".format(tbl=OBL_TRANSACTIONS_ACTOR_COUNTS)
#s = pd.read_sql_query(query, con)
#s.to_csv(OBL_TRANSACTIONS_ACTOR_COUNTS+".csv", sep=",", index=False, encoding="utf-8")

In [32]:
con.close()